# Deploy a GenAI Chatbot to Cloud Run

A step-by-step Colab notebook for deploying a **Flask + Gemini 3.1 Pro** chatbot to **Google Cloud Run**.

**Course Project ID:** `statg529300220261-01d2`

---

## What you'll learn

1. How to enable the required Google Cloud APIs
2. How to write the app files (Flask + Gemini + Dockerfile)
3. How to deploy to Cloud Run with a single command
4. How to test the live endpoint
5. How to clean up to avoid charges

---

## What is Cloud Run?

| Feature | Description |
|---|---|
| **Serverless** | No servers to manage — Google handles scaling and infrastructure |
| **Scale to zero** | You only pay when your app is handling requests |
| **HTTPS by default** | Every service gets a unique `*.run.app` URL with TLS |
| **Source deploy** | Deploy directly from source code — Cloud Run builds the container for you |

---

## Step 1: Authenticate with Google Cloud

Sign in with your **school email** (the one that has access to the course project).

> **Important:** You must sign in with the account that has access to project `statg529300220261-01d2`.

In [ ]:
from google.colab import auth

# This will open a pop-up asking you to sign in with your Google account.
# Sign in with your school email (e.g., your_uni@columbia.edu).
auth.authenticate_user()

print("Authentication successful!")

### Set your Project ID

Set the project ID for the course. If you have a different project, replace the value below.

In [ ]:
# ============================================================
# CONFIGURATION — Update this if your project ID is different
# ============================================================
PROJECT_ID = "statg529300220261-01d2"
REGION = "us-central1"
SERVICE_NAME = "gemini-chatbot"

print(f"Project ID   : {PROJECT_ID}")
print(f"Region       : {REGION}")
print(f"Service name : {SERVICE_NAME}")

!gcloud config set project {PROJECT_ID}

---

## Step 2: Enable Required APIs

Cloud Run needs several APIs enabled: Cloud Run, Cloud Build (to build the container), Artifact Registry (to store the container image), and Vertex AI (for Gemini).

In [ ]:
!gcloud services enable \
  run.googleapis.com \
  cloudbuild.googleapis.com \
  artifactregistry.googleapis.com \
  aiplatform.googleapis.com

print("\nAll required APIs are enabled.")

---

## Step 3: Grant Vertex AI Permissions to Cloud Run

When your app runs on Cloud Run, it uses a **service account** (not your personal credentials). We need to grant this service account permission to call Vertex AI.

In [ ]:
import subprocess

# Get the project number
result = subprocess.run(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
    capture_output=True, text=True
)
PROJECT_NUMBER = result.stdout.strip()
SERVICE_ACCOUNT = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"

print(f"Project number  : {PROJECT_NUMBER}")
print(f"Service account : {SERVICE_ACCOUNT}")

In [ ]:
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
  --member="serviceAccount:{SERVICE_ACCOUNT}" \
  --role="roles/aiplatform.user" \
  --quiet

print(f"\nGranted roles/aiplatform.user to {SERVICE_ACCOUNT}")

---

## Step 4: Write the App Files

We'll create the Flask app, HTML template, requirements, and Dockerfile directly in this notebook.

> **Alternative:** If you've already cloned the repo, you can skip this step and `cd` into `cloud_run/app/` instead.

In [ ]:
import os

APP_DIR = "gemini-chatbot"
os.makedirs(f"{APP_DIR}/templates", exist_ok=True)

print(f"Created directory: {APP_DIR}/templates/")

### `main.py` — Flask app with Gemini API

In [ ]:
%%writefile {APP_DIR}/main.py
import os

from flask import Flask, jsonify, render_template, request
from google import genai

app = Flask(__name__)

# ── Configuration ────────────────────────────────────────────
PROJECT_ID = os.environ.get("PROJECT_ID", "statg529300220261-01d2")
LOCATION = "global"
MODEL_ID = "gemini-3.1-pro-preview"

# Initialize the Vertex AI client once at startup
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)


@app.route("/")
def index():
    """Serve the chat web UI."""
    return render_template("index.html")


@app.route("/api/chat", methods=["POST"])
def chat():
    """Send a user message to Gemini and return the response."""
    data = request.get_json(silent=True) or {}
    message = data.get("message", "").strip()
    if not message:
        return jsonify({"error": "message is required"}), 400

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=message,
        )
        return jsonify({"reply": response.text})
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8080))
    app.run(host="0.0.0.0", port=port, debug=True)

### `templates/index.html` — Chat web UI

In [ ]:
%%writefile {APP_DIR}/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Gemini Chatbot</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
      background: #f5f5f5;
      display: flex; flex-direction: column; height: 100vh;
    }
    header { background: #1a73e8; color: #fff; padding: 16px 24px; font-size: 18px; font-weight: 600; }
    #chat { flex: 1; overflow-y: auto; padding: 24px; display: flex; flex-direction: column; gap: 12px; }
    .msg { max-width: 70%; padding: 12px 16px; border-radius: 16px; line-height: 1.5; white-space: pre-wrap; word-wrap: break-word; }
    .msg.user { align-self: flex-end; background: #1a73e8; color: #fff; border-bottom-right-radius: 4px; }
    .msg.bot { align-self: flex-start; background: #fff; color: #202124; border: 1px solid #e0e0e0; border-bottom-left-radius: 4px; }
    .msg.error { align-self: flex-start; background: #fce8e6; color: #c5221f; border: 1px solid #f5c6cb; border-bottom-left-radius: 4px; }
    .typing { align-self: flex-start; color: #999; font-style: italic; padding: 8px 16px; }
    #input-area { display: flex; padding: 16px 24px; background: #fff; border-top: 1px solid #e0e0e0; gap: 12px; }
    #message { flex: 1; padding: 12px 16px; font-size: 16px; border: 1px solid #dadce0; border-radius: 24px; outline: none; }
    #message:focus { border-color: #1a73e8; }
    #send { padding: 12px 24px; font-size: 16px; background: #1a73e8; color: #fff; border: none; border-radius: 24px; cursor: pointer; }
    #send:disabled { opacity: 0.5; cursor: not-allowed; }
    #send:hover:not(:disabled) { background: #1765cc; }
  </style>
</head>
<body>
  <header>Gemini Chatbot — Cloud Run Demo</header>
  <div id="chat"></div>
  <div id="input-area">
    <input type="text" id="message" placeholder="Type a message..." autocomplete="off">
    <button id="send">Send</button>
  </div>
  <script>
    const chat = document.getElementById("chat");
    const input = document.getElementById("message");
    const sendBtn = document.getElementById("send");
    function addMessage(text, cls) {
      const div = document.createElement("div");
      div.className = "msg " + cls;
      div.textContent = text;
      chat.appendChild(div);
      chat.scrollTop = chat.scrollHeight;
      return div;
    }
    async function send() {
      const text = input.value.trim();
      if (!text) return;
      addMessage(text, "user");
      input.value = "";
      sendBtn.disabled = true;
      const typing = document.createElement("div");
      typing.className = "typing";
      typing.textContent = "Gemini is thinking...";
      chat.appendChild(typing);
      chat.scrollTop = chat.scrollHeight;
      try {
        const res = await fetch("/api/chat", {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ message: text }),
        });
        const data = await res.json();
        typing.remove();
        if (res.ok) { addMessage(data.reply, "bot"); }
        else { addMessage("Error: " + (data.error || "Unknown error"), "error"); }
      } catch (err) {
        typing.remove();
        addMessage("Network error: " + err.message, "error");
      } finally {
        sendBtn.disabled = false;
        input.focus();
      }
    }
    sendBtn.addEventListener("click", send);
    input.addEventListener("keydown", (e) => { if (e.key === "Enter" && !e.shiftKey) { e.preventDefault(); send(); } });
    input.focus();
  </script>
</body>
</html>

### `requirements.txt`

In [ ]:
%%writefile {APP_DIR}/requirements.txt
Flask~=3.1
gunicorn~=23.0
google-genai>=1.0

### `Dockerfile`

In [ ]:
%%writefile {APP_DIR}/Dockerfile
FROM python:3.12-slim

# Prevent Python from buffering stdout/stderr (better for Cloud Run logs)
ENV PYTHONUNBUFFERED=1

WORKDIR /app

# Install dependencies first (layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code
COPY . .

# Cloud Run injects the PORT env var (default 8080)
CMD exec gunicorn --bind :${PORT:-8080} --workers 1 --threads 8 --timeout 0 main:app

### Verify the files

In [ ]:
import os

for root, dirs, files in os.walk(APP_DIR):
    level = root.replace(APP_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

---

## Step 5: Deploy to Cloud Run

Deploy directly from source — Cloud Run will build the container, push it to Artifact Registry, and create the service.

> **Note:** The first deploy takes 2-3 minutes while the container is built.
>
> Our organization policy does not allow public (`allUsers`) access to Cloud Run services, so we deploy **without** `--allow-unauthenticated` and grant domain-level access separately.

In [ ]:
%cd {APP_DIR}

!gcloud run deploy {SERVICE_NAME} \
  --source . \
  --region {REGION} \
  --set-env-vars PROJECT_ID={PROJECT_ID} \
  --quiet

### Get the Service URL

In [ ]:
import subprocess

result = subprocess.run(
    ["gcloud", "run", "services", "describe", SERVICE_NAME,
     "--region", REGION, "--format=value(status.url)"],
    capture_output=True, text=True
)
SERVICE_URL = result.stdout.strip()

print(f"Your chatbot is live at: {SERVICE_URL}")
print(f"\nOpen this URL in your browser to use the chat UI.")

### Grant access to your organization

Our Google Cloud org policy does not allow public (`allUsers`) access. Instead, grant invoke access to the `columbia.edu` domain so anyone with a school Google account can reach the chatbot.

In [ ]:
!gcloud run services add-iam-policy-binding {SERVICE_NAME} \
  --region {REGION} \
  --member="domain:columbia.edu" \
  --role="roles/run.invoker"

print(f"\nGranted roles/run.invoker to domain:columbia.edu")

---

## Step 6: Open the Chat UI in Your Browser

Since our organization policy requires authentication, you **cannot** open the Cloud Run URL directly in a browser — you'll get a "Forbidden" error. Instead, use the **Cloud Run Proxy** to access the chat UI from your laptop.

### How it works

The proxy runs on your laptop and forwards browser requests to Cloud Run, automatically attaching your `gcloud` credentials. You see the app at `http://localhost:8080` as if it were running locally, but it's actually your deployed Cloud Run service.

### Instructions

1. Open **Terminal** on your laptop (not Colab)
2. Run the following command:

```
gcloud run services proxy gemini-chatbot --region us-central1 --port 8080
```

3. The first time, gcloud will ask to install the `cloud-run-proxy` component — type **Y**
4. Once you see `http://127.0.0.1:8080 proxies to https://gemini-chatbot-...run.app`, open **http://localhost:8080** in your browser
5. You should see the Gemini Chatbot UI — type a message and press Enter!
6. Press **Ctrl+C** in Terminal when you're done

> **Note:** The proxy must stay running in Terminal while you use the chat UI. If you close Terminal, the proxy stops and `localhost:8080` will no longer work.

In [ ]:
# Copy-paste this command into your laptop Terminal:
print("=" * 60)
print("  Run this in your laptop Terminal (not Colab):")
print("=" * 60)
print()
print(f"  gcloud run services proxy {SERVICE_NAME} --region {REGION} --port 8080")
print()
print("  Then open: http://localhost:8080")
print("=" * 60)

---

## Step 7: Test the API Endpoint from Colab

You can also test the `/api/chat` endpoint directly from Colab using `curl` with an identity token.

In [ ]:
%%bash -s "$SERVICE_URL"
SERVICE_URL=$1

echo "Testing: POST $SERVICE_URL/api/chat"
echo "---"

curl -s -X POST "$SERVICE_URL/api/chat" \
  -H "Authorization: Bearer $(gcloud auth print-identity-token)" \
  -H "Content-Type: application/json" \
  -d '{"message": "What is Cloud Run? Explain in 2 sentences."}' | python3 -m json.tool

### Try it yourself

Change the message in the JSON below and run the cell to ask Gemini anything:

In [ ]:
%%bash -s "$SERVICE_URL"
SERVICE_URL=$1

curl -s -X POST "$SERVICE_URL/api/chat" \
  -H "Authorization: Bearer $(gcloud auth print-identity-token)" \
  -H "Content-Type: application/json" \
  -d '{"message": "What are 3 benefits of serverless computing?"}' | python3 -m json.tool

---

## Step 8: Clean Up

Delete the Cloud Run service to avoid any charges.

> **Important:** Always clean up when you're done experimenting.

In [ ]:
!gcloud run services delete {SERVICE_NAME} --region {REGION} --quiet

print(f"\nService '{SERVICE_NAME}' has been deleted.")

Optionally, delete the container images from Artifact Registry:

In [ ]:
# Uncomment the line below to delete the container repository
# !gcloud artifacts repositories delete cloud-run-source-deploy --location {REGION} --quiet

---

## Troubleshooting

### "Forbidden — Your client does not have permission to get URL /"
The service requires authentication. Our org policy blocks public (`allUsers`) access.
- Make sure you ran the IAM binding in Step 5 (`domain:columbia.edu` with `roles/run.invoker`)
- When calling from code, include an identity token: `Authorization: Bearer $(gcloud auth print-identity-token)`
- From your laptop, use the Cloud Run proxy: `gcloud run services proxy gemini-chatbot --region us-central1 --port 8080`

### "FAILED_PRECONDITION: One or more users named in the policy do not belong to a permitted customer"
You tried to grant access to `allUsers` or `allAuthenticatedUsers`. Use `domain:columbia.edu` instead.

### "Permission denied" calling Vertex AI from Cloud Run
- Re-run Step 3 to grant `roles/aiplatform.user` to the service account

### Build fails
- Make sure all 4 files (`main.py`, `requirements.txt`, `Dockerfile`, `templates/index.html`) were created in Step 4
- Re-run the `%%writefile` cells if needed

### App deploys but returns 500 errors
Check the logs:
```
!gcloud run services logs read gemini-chatbot --region us-central1 --limit 20
```

### "Cloud Run Admin API has not been enabled"
- Re-run Step 2 to enable all required APIs

### Links

- [Cloud Run Documentation](https://cloud.google.com/run/docs)
- [Vertex AI GenAI SDK](https://cloud.google.com/vertex-ai/generative-ai/docs/sdks/overview)
- [Gemini 3.1 Pro in Model Garden](https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/gemini-3.1-pro-preview)